In [4]:
%run init_env.py

Added to Python path: D:\git\cs5305\genai-langchain
Environment initialization completed successfully.


## Custom state schema

```python
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str
```

- `AgentState` is imported as the base state type used by the agent.
- `CustomState` extends `AgentState` to define extra data the agent should keep track of.
- `favourite_colour: str` adds a new field to the agent state for storing the user's favourite colour.
- This schema is later passed to `create_agent(..., state_schema=CustomState)` so the agent knows that `favourite_colour` is part of its state.

In [7]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

# CommandObject

The `Command` object is used in LangGraph to update agent state and control the flow of execution. Here's how it's used in the `update_favourite_colour` tool:

```python
return Command(update={
    "favourite_colour": favourite_colour, 
    "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
)
```

**Key components:**

- **`update` parameter**: A dictionary that specifies which state fields to update and their new values
  - `"favourite_colour": favourite_colour` – stores the user's colour preference in agent state
  - `"messages": [...]` – appends a `ToolMessage` back to the conversation thread

- **`ToolMessage`**: Represents the tool's response, containing:
  - `content`: The message text ("Successfully updated favourite colour")
  - `tool_call_id`: Links the response back to the original tool invocation (from `runtime.tool_call_id`)

This allows tools to both modify internal state AND communicate results back to the LLM, creating a feedback loop for stateful agent interactions.

In [8]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

# Command objects are used to update the state of the agent, and can also include messages that will be sent back to the user. 
# In this case, we're updating the favourite colour in the state, and also sending a message back to the user confirming that 
# we've updated it.
@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    create_azure_llm(),
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

Creating Azure OpenAI LLM
  Deployment: lums-gpt-4.1-mini
  Endpoint: https://aoai-foundry-swc.openai.azure.com/
  API Version: 2025-01-01-preview
  Temperature: 0.7


In [10]:
from pprint import pprint
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='4e309ce3-cd52-4d02-9187-7d6ef9e80cf4'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 60, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DSmD0EogUkq5jLoQgBqJ8tnVzfaZ4', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'},

### Explanation of this cell

```python
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)
```

- `agent.invoke(...)` runs one interaction with the configured agent.
- The first argument is the **input state**:
  - `messages`: starts the conversation with a user message (`"Hello, how are you?"`).
  - `favourite_colour`: preloads state with `"green"` for this run.
- The second argument sets runtime config:
  - `thread_id: "10"` creates/uses a separate conversation thread (isolated memory for that thread).
- The result is stored in `response`.
- `pprint(response)` prints the returned state/messages in a readable format.

**Why this matters:** even though the user message is just a greeting, the state already contains `favourite_colour="green"`, so tools or later turns in the same thread can use it.

In [11]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='9a01da7d-0423-45b6-a1ae-32f028978d6f'),
              AIMessage(content="Hello! I'm doing great, thank you for asking. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 61, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DSmDVGj4FzjaYfIZlP6vGcFpV1WBi', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severi

## Read state

In [12]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    create_azure_llm(),
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

Creating Azure OpenAI LLM
  Deployment: lums-gpt-4.1-mini
  Endpoint: https://aoai-foundry-swc.openai.azure.com/
  API Version: 2025-01-01-preview
  Temperature: 0.7


## End-to-end state demo 

1. **Store user preference**  
    The agent is invoked with:  
    `"My favourite colour is green"`  
    This triggers `update_favourite_colour`, which writes `favourite_colour` into agent state.
    The agent uses the LLM's ability to recognize when a tool should be called based on the conversation context. Here's what happens:

    1. **LLM receives the message** `"My favourite colour is green"`
    2. **LLM analyzes available tools** - it sees `update_favourite_colour` with description: "Update the favourite colour of the user in the state once they've revealed it."
    3. **LLM matches intent to tool** - the message contains a colour preference, which matches the tool's purpose
    4. **LLM generates a tool call** with the extracted parameter (`favourite_colour="green"`)
    5. **LangGraph executes the tool** and updates the state
    6. **Tool result is returned** to the LLM, which can then respond to the user

    Modern LLMs are trained on tool-use patterns and can reliably recognize when to invoke tools based on function descriptions and the conversation context.

In [13]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='2a62622e-c188-46a9-b64e-edfe8a27ea56'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 81, 'total_tokens': 100, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DSmEnTzqRaS6LJSkn24k09OC4TvI3', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}

In [14]:
pprint(response['messages'][1].tool_calls[0])

{'args': {'favourite_colour': 'green'},
 'id': 'call_zhkjbzssEys5jqDdnXEV3mWj',
 'name': 'update_favourite_colour',
 'type': 'tool_call'}


**Read user preference**  
    The agent is invoked again with:  
    `"What's my favourite colour?"`  
    using the same `thread_id="1"`, so prior state is available.  
    The `read_favourite_colour` tool returns the stored value (`"green"`).

### Important note
State continuity depends on reusing the same thread identifier in `configurable.thread_id`.  

In [15]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='2a62622e-c188-46a9-b64e-edfe8a27ea56'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 81, 'total_tokens': 100, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DSmEnTzqRaS6LJSkn24k09OC4TvI3', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}